<a href="https://colab.research.google.com/github/MohammedAyman07/curriculum-rag-research/blob/main/notebooks/03_dense_retrieval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CurriculumRAG — Dense Semantic Retrieval

## Objective
Implement semantic document retrieval using a pretrained
Sentence Transformers model.

## Model
sentence-transformers/all-MiniLM-L6-v2

## Methodology
1. Load the educational passage collection.
2. Generate document embeddings.
3. Generate question embeddings.
4. Calculate cosine similarity.
5. Retrieve the top-five passages.

## Evaluation
- Precision@5
- Recall@5
- Mean Reciprocal Rank (MRR)
- Query latency

In [1]:

# CurriculumRAG - Dense Retrieval Environment

%pip -q install -U sentence-transformers

import numpy as np
import pandas as pd
import torch
import time

from pathlib import Path
from sentence_transformers import SentenceTransformer

print("Dense retrieval environment ready!")
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


Dense retrieval environment ready!
PyTorch version: 2.14.1+cpu
CUDA available: False


In [2]:

# Load the same research dataset used by TF-IDF

BASE_URL = (
    "https://raw.githubusercontent.com/"
    "MohammedAyman07/curriculum-rag-research/main/data/"
)

passages_df = pd.read_csv(BASE_URL + "passages.csv")
queries_df = pd.read_csv(BASE_URL + "queries.csv")

assert len(passages_df) == 40
assert len(queries_df) == 60

assert passages_df["doc_id"].is_unique
assert queries_df["query_id"].is_unique

documents = passages_df["text"].fillna("").astype(str).tolist()
document_ids = passages_df["doc_id"].astype(str).tolist()

dev_queries = queries_df[
    queries_df["split"] == "dev"
].copy()

assert len(dev_queries) == 15

print("Dataset loaded successfully!")
print("Passages:", len(passages_df))
print("Questions:", len(queries_df))
print("Development questions:", len(dev_queries))
print("Test questions: 45 (held out)")


Dataset loaded successfully!
Passages: 40
Questions: 60
Development questions: 15
Test questions: 45 (held out)


In [3]:

# Load Sentence Transformers model

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

model_load_start = time.perf_counter()

model = SentenceTransformer(
    MODEL_NAME,
    device=DEVICE
)

model_load_time = time.perf_counter() - model_load_start

print("Model loaded successfully!")
print("Model:", MODEL_NAME)
print("Device:", DEVICE)
print(f"Model loading time: {model_load_time:.3f} seconds")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully!
Model: sentence-transformers/all-MiniLM-L6-v2
Device: cpu
Model loading time: 9.830 seconds


In [4]:

# Generate embeddings for all educational passages

embedding_start = time.perf_counter()

if hasattr(model, "encode_document"):
    document_embeddings = model.encode_document(
        documents,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )
else:
    document_embeddings = model.encode(
        documents,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

document_embeddings = np.asarray(
    document_embeddings,
    dtype=np.float32
)

document_embedding_time = (
    time.perf_counter() - embedding_start
)

assert document_embeddings.shape[0] == len(documents)

print("\nDocument embeddings generated!")
print("Embedding matrix shape:", document_embeddings.shape)
print("Embedding dimension:", document_embeddings.shape[1])
print(
    f"Document embedding time: "
    f"{document_embedding_time:.3f} seconds"
)


Batches:   0%|          | 0/2 [00:00<?, ?it/s]


Document embeddings generated!
Embedding matrix shape: (40, 384)
Embedding dimension: 384
Document embedding time: 2.640 seconds


In [5]:

# Dense semantic retrieval function

def retrieve_dense(question, top_k=5):

    start = time.perf_counter()

    if hasattr(model, "encode_query"):
        query_embedding = model.encode_query(
            [question],
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False
        )[0]
    else:
        query_embedding = model.encode(
            [question],
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False
        )[0]

    query_embedding = np.asarray(
        query_embedding,
        dtype=np.float32
    )

    # Normalized vectors allow dot product to represent cosine similarity
    similarity_scores = document_embeddings @ query_embedding

    ranked_indices = np.argsort(
        -similarity_scores,
        kind="stable"
    )[:top_k]

    results = []

    for index in ranked_indices:
        results.append({
            "doc_id": document_ids[index],
            "score": float(similarity_scores[index]),
            "topic": passages_df.iloc[index]["topic"],
            "text": documents[index]
        })

    latency_ms = (
        time.perf_counter() - start
    ) * 1000

    return results, latency_ms


# Test the retrieval function

sample_question = "How does binary search reduce the search space?"

sample_results, sample_latency = retrieve_dense(sample_question)

print("Question:", sample_question)
print(f"Query latency: {sample_latency:.3f} ms\n")

for rank, result in enumerate(sample_results, start=1):

    print(f"Rank {rank}: {result['doc_id']}")
    print("Topic:", result["topic"])
    print("Similarity:", round(result["score"], 4))
    print()


Question: How does binary search reduce the search space?
Query latency: 129.076 ms

Rank 1: P033
Topic: Binary Search
Similarity: 0.6899

Rank 2: P014
Topic: Binary Search Trees
Similarity: 0.5122

Rank 3: P032
Topic: Linear Search
Similarity: 0.4138

Rank 4: P002
Topic: Space Complexity
Similarity: 0.413

Rank 5: P037
Topic: Linear Probing
Similarity: 0.4125



In [6]:

# Evaluation metrics

def calculate_metrics(retrieved_ids, relevant_ids, k=5):

    retrieved = retrieved_ids[:k]
    relevant = set(relevant_ids)

    if not relevant:
        raise ValueError("Each question needs at least one relevant passage.")

    hits = set(retrieved) & relevant

    precision = len(hits) / k
    recall = len(hits) / len(relevant)

    reciprocal_rank = 0.0

    for rank, doc_id in enumerate(retrieved, start=1):

        if doc_id in relevant:
            reciprocal_rank = 1 / rank
            break

    return precision, recall, reciprocal_rank


# Warm up the model before timing
retrieve_dense("What is a binary search tree?")


evaluation_records = []

for _, row in dev_queries.iterrows():

    results, _ = retrieve_dense(
        row["question"],
        top_k=5
    )

    retrieved_ids = [
        result["doc_id"] for result in results
    ]

    relevant_ids = str(
        row["relevant_doc_ids"]
    ).split("|")

    precision, recall, mrr = calculate_metrics(
        retrieved_ids,
        relevant_ids,
        k=5
    )

    # Measure query latency using repeated runs
    latency_samples = [
        retrieve_dense(row["question"], top_k=5)[1]
        for _ in range(5)
    ]

    evaluation_records.append({
        "query_id": row["query_id"],
        "question": row["question"],
        "relevant_doc_ids": row["relevant_doc_ids"],
        "retrieved_doc_ids": "|".join(retrieved_ids),
        "precision_at_5": precision,
        "recall_at_5": recall,
        "mrr": mrr,
        "median_query_latency_ms": float(
            np.median(latency_samples)
        )
    })


dense_results_df = pd.DataFrame(evaluation_records)

print("Dense retrieval development evaluation completed!")

display(dense_results_df)


Dense retrieval development evaluation completed!


,query_id,question,relevant_doc_ids,retrieved_doc_ids,precision_at_5,recall_at_5,mrr,median_query_latency_ms
0,Q001,How does an algorithm's running time change as...,P001,P001|P002|P033|P025|P004,0.2,1.0,1.0,21.014521
1,Q004,Why can an array element be accessed directly ...,P004,P004|P034|P027|P026|P037,0.2,1.0,1.0,38.364470
2,Q006,Which linked list supports traversal in both d...,P006,P005|P006|P007|P016|P036,0.2,1.0,0.5,35.506449
3,Q008,Which principle explains why the latest stack ...,P008,P008|P009|P030|P006|P025,0.2,1.0,1.0,30.864616
4,Q010,Where does a queue insert new elements and rem...,P010,P010|P011|P012|P008|P004,0.2,1.0,1.0,39.860292
5,Q013,How many children can a node have in a binary ...,P013,P013|P014|P015|P016|P033,0.2,1.0,1.0,43.150161
6,Q015,What balance condition does an AVL tree mainta...,P015,P015|P014|P016|P021|P023,0.2,1.0,1.0,43.744944
7,Q018,"What are the main components of a graph, and h...",P017,P017|P018|P019|P021|P023,0.2,1.0,1.0,34.006311
8,Q020,Which graph traversal visits nearby vertices b...,P019,P019|P020|P016|P017|P018,0.2,1.0,1.0,54.122829
9,Q022,How does DFS explore a graph and then return t...,P020,P020|P019|P018|P017|P016,0.2,1.0,1.0,26.010966


In [7]:

# Calculate dense retrieval performance

dense_summary = {
    "Method": "Dense Semantic Retrieval",
    "Model": MODEL_NAME,
    "Device": DEVICE,
    "Precision@5": dense_results_df["precision_at_5"].mean(),
    "Recall@5": dense_results_df["recall_at_5"].mean(),
    "MRR": dense_results_df["mrr"].mean(),
    "Median Query Latency (ms)": (
        dense_results_df["median_query_latency_ms"].median()
    ),
    "Model Loading Time (seconds)": model_load_time,
    "Document Embedding Time (seconds)": document_embedding_time
}

dense_summary_df = pd.DataFrame([dense_summary])

display(dense_summary_df)

# Save experiment outputs
RESULTS_DIR = Path("/content/curriculum-rag-results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

dense_results_df.to_csv(
    RESULTS_DIR / "dense_dev_predictions.csv",
    index=False
)

dense_summary_df.to_csv(
    RESULTS_DIR / "dense_dev_metrics.csv",
    index=False
)

print("Dense retrieval results saved successfully!")


,Method,Model,Device,Precision@5,Recall@5,MRR,Median Query Latency (ms),Model Loading Time (seconds),Document Embedding Time (seconds)
0,Dense Semantic Retrieval,sentence-transformers/all-MiniLM-L6-v2,cpu,0.2,1.0,0.966667,30.864616,9.830236,2.639891


Dense retrieval results saved successfully!
